# AUD/USD Technical ML Model

## خصائص AUD/USD وتأثيرها على الباراميترات:

| الخاصية | EUR/USD | AUD/USD | التأثير |
|---|---|---|---|
| **Volatility** | أعلى | أقل ~20% | ATR_MULT أقل، PIP_THRESHOLD أقل |
| **أهم session** | London/NY | **Sydney** | session_momentum: Sydney أعلى وزن |
| **نوع العملة** | Major | **Commodity** | يتأثر بالذهب + China data |
| **البنك المركزي** | ECB | **RBA** | خطر أول ثلاثاء كل شهر |
| **Spread** | 2 pip | 3 pip | SPREAD_BPS = 3 |
| **Trend cycles** | 3-4 أشهر | 4-6 أشهر | years أعلى للـ 4h/1d |
| **Mean reversion** | متوسط | أعلى في Short TF | gamma أعلى في 5min/15min |

**ترتيب التشغيل:** شغّل Cell 1 (training) ثم Cell 2 (download)


In [ ]:
import requests
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import TimeSeriesSplit
import joblib
import os
import json
from datetime import datetime, timedelta
import time
import random
import logging
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)

# ─────────────────────────────────────────────
# ★ المتغيرات الأساسية
# ─────────────────────────────────────────────
TWELVE_DATA_API_KEY = os.environ.get("TWELVE_DATA_API_KEY", "")
SYMBOL       = "AUD/USD"
SYMBOL_SAFE  = "AUDUSD"
CSV_RAW_FILE = f"{SYMBOL_SAFE}_5min_raw.csv"
MODELS_DIR   = "models/audusd"

ATR_MULT_STOP = 1.4  # AUD: أقل volatility من EUR
ATR_MULT_TAKE = 2.2  # AUD: أقل volatility
SPREAD_BPS    = 3  # AUD/USD: spread أعلى قليلاً من EUR/USD
PIP_SIZE      = 0.0001

# ══════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════════

# pip threshold per timeframe — minimum movement to count as signal
PIP_THRESHOLD = {
    
    "5min":  3,    # كان 6 — أكثر samples، ATR~10pip
    "15min": 8,    # ثابت (ناجح)
    "45min": 12,   # ثابت
    "1h":    15,   # ثابت
    "2h":    20,   # ثابت
    "4h":    28,   # ثابت
    "1d":    45,   # ثابت
    "1W":    15,   # كان 80 — صفقات أكثر في الـ holdout
}

# horizon per timeframe
HORIZON = {
    # ★ FIX 5min: كان 5 (25 دقيقة) → 3 (15 دقيقة) — أوضح وأقل ambiguity
    "5min":  3,
    "15min": 2,
    "45min": 2,
    "1h":    1,
    "2h":    1,
    "4h":    1,
    "1d":    1,
    "1W":    2,
}

TIMEFRAME_CONFIG = {
    # ══════════════════════════════════════════════════════
    
    # ══════════════════════════════════════════════════════
    "5min": {
        "interval": "5min", "years": 6,
        "buy_threshold": 0.58, "sell_threshold": 0.42,
        "fold_buy_threshold": 0.56, "fold_sell_threshold": 0.44,
        "max_trades_per_day": 4,   # كان 3 → رفع لزيادة الصفقات
        "resample_rule": "5min", "larger_tf": "1h",
        "htf_features": ["rsi","trend_filter","macd_hist","market_structure","session_momentum","atr_pct","adx"],
        "minutes_per_candle": 5, "min_candles": 500,
        "adx_min_entry": 15,       # ★ FIX: كان 22 → 15 (أقل تقييداً)
        "session_filter": False,   # ★ FIX: إيقاف — يمنع صفقات Asia مهمة لـ AUD
        "atr_mult_stop": 1.5, "atr_mult_take": 2.8,
    },
    "15min": {
        "interval": "15min", "years": 3,
        "buy_threshold": 0.60, "sell_threshold": 0.40,
        "fold_buy_threshold": 0.56, "fold_sell_threshold": 0.44,
        "max_trades_per_day": 4,   # EUR=5
        "resample_rule": "15min",
        "larger_tf": "4h",
        "htf_features": ["rsi", "trend_filter", "atr_pct", "market_structure", "session_momentum", "ema_alignment"],
        "minutes_per_candle": 15, "min_candles": 350
    },
    "45min": {
        "interval": "45min", "years": 3,
        "buy_threshold": 0.58, "sell_threshold": 0.42,
        "fold_buy_threshold": 0.55, "fold_sell_threshold": 0.45,
        "max_trades_per_day": None,
        "resample_rule": "45min",
        "larger_tf": "4h",
        "htf_features": ["rsi", "trend_filter", "macd_hist", "ema_alignment", "atr_pct"],
        "minutes_per_candle": 45, "min_candles": 180
    },
    "1h": {
        "interval": "1h", "years": 5,  # EUR=4 — AUD نحتاج بيانات أكثر
        "buy_threshold": 0.60, "sell_threshold": 0.40,
        "fold_buy_threshold": 0.55, "fold_sell_threshold": 0.45,
        "max_trades_per_day": None,
        "resample_rule": "1h",
        "larger_tf": "4h",
        "htf_features": ["rsi", "trend_filter", "market_structure", "sr_dist", "ema_alignment", "atr_pct"],
        "minutes_per_candle": 60, "min_candles": 200
    },
    "2h": {
        "interval": "2h", "years": 5,  # EUR=4
        "buy_threshold": 0.57, "sell_threshold": 0.43,
        "fold_buy_threshold": 0.54, "fold_sell_threshold": 0.46,
        "max_trades_per_day": None,
        "resample_rule": "2h",
        "larger_tf": "1d",
        "htf_features": ["rsi", "trend_filter", "atr_pct", "ema_alignment", "market_structure"],
        "minutes_per_candle": 120, "min_candles": 120
    },
    "4h": {
        "interval": "4h", "years": 7,  # EUR=5 — AUD trend cycles أطول
        "buy_threshold": 0.57, "sell_threshold": 0.43,
        "fold_buy_threshold": 0.54, "fold_sell_threshold": 0.46,
        "max_trades_per_day": None,
        "resample_rule": "4h",
        "larger_tf": "1d",
        "htf_features": ["rsi", "trend_filter", "market_structure", "sr_dist", "ema_alignment", "atr_pct"],
        # AUD/USD: اقل من EUR — volatility أقل
        "atr_mult_stop": 1.6, "atr_mult_take": 2.6,
        "minutes_per_candle": 240, "min_candles": 80
    },
    "1d": {
        "interval": "1d", "years": 20,
        "buy_threshold": 0.55, "sell_threshold": 0.45,
        "fold_buy_threshold": 0.54, "fold_sell_threshold": 0.46,
        "max_trades_per_day": None,
        "resample_rule": "1B",
        "larger_tf": "1W",
        "htf_features": ["rsi", "trend_filter", "ema_alignment", "market_structure"],
        "atr_mult_stop": 1.8, "atr_mult_take": 2.8,
        "minutes_per_candle": 1440, "min_candles": 80
    },
    "1W": {
        "interval": "1week",
        "years": 25,                    # زد البيانات
        "buy_threshold": 0.52,          # خفضناه عشان يدخل صفقات أكثر
        "sell_threshold": 0.48,
        "fold_buy_threshold": 0.53,
        "fold_sell_threshold": 0.47,
        "max_trades_per_day": None,
        "resample_rule": "W-FRI",
        "larger_tf": None,
        "htf_features": ["rsi","trend_filter","ema_alignment","market_structure"],
        "minutes_per_candle": 10080,
        "min_candles": 40,
        "adx_min_entry": 0,
        "session_filter": False,
        "atr_mult_stop": 2.6,           # خفضناه شوية
        "atr_mult_take": 4.2,
    }
}


# ══════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════════
XGB_PARAMS = {
    # AUD/USD XGB params:
    # - 5min/15min: regularization أقوى قليلاً (mean-reverting أكثر)
    # - 1h/4h: يمكن مزيد من الـ depth (signals أوضح)
    # - الباقي: نفس EUR/USD تقريباً
    "5min": {
        "max_depth": 4, "learning_rate": 0.018, "gamma": 0.55,
        "min_child_weight": 15, "reg_alpha": 1.2, "reg_lambda": 3.2,
        "subsample": 0.55, "colsample_bytree": 0.55,
        "n_estimators": 600
    },
    "15min": {
        "max_depth": 4, "learning_rate": 0.022, "gamma": 0.42,
        "min_child_weight": 12, "reg_alpha": 0.85, "reg_lambda": 2.6,
        "subsample": 0.60, "colsample_bytree": 0.60,
        "n_estimators": 700
    },
    "45min": {
        "max_depth": 5, "learning_rate": 0.028, "gamma": 0.25,
        "min_child_weight": 8, "reg_alpha": 0.50, "reg_lambda": 1.8,
        "subsample": 0.65, "colsample_bytree": 0.65,
        "n_estimators": 850
    },
    "1h": {
        "max_depth": 5, "learning_rate": 0.022, "gamma": 0.20,
        "min_child_weight": 7, "reg_alpha": 0.35, "reg_lambda": 1.5,
        "subsample": 0.70, "colsample_bytree": 0.70,
        "n_estimators": 950
    },
    "2h": {
        "max_depth": 6, "learning_rate": 0.020, "gamma": 0.15,
        "min_child_weight": 6, "reg_alpha": 0.25, "reg_lambda": 1.3,
        "subsample": 0.72, "colsample_bytree": 0.72,
        "n_estimators": 1000
    },
    "4h": {
        "max_depth": 6, "learning_rate": 0.016, "gamma": 0.12,
        "min_child_weight": 5, "reg_alpha": 0.20, "reg_lambda": 1.2,
        "subsample": 0.75, "colsample_bytree": 0.75,
        "n_estimators": 1100
    },
    "1d": {
        "max_depth": 6, "learning_rate": 0.014, "gamma": 0.10,
        "min_child_weight": 4, "reg_alpha": 0.15, "reg_lambda": 1.0,
        "subsample": 0.78, "colsample_bytree": 0.78,
        "n_estimators": 1200
    },
    "1W": {
        "max_depth": 5,
        "learning_rate": 0.015,
        "gamma": 0.08,
        "min_child_weight": 3,
        "reg_alpha": 0.08,
        "reg_lambda": 0.7,
        "subsample": 0.82,
        "colsample_bytree": 0.80,
        "n_estimators": 900
    },
    "default": {
        "max_depth": 5, "learning_rate": 0.02, "gamma": 0.20,
        "min_child_weight": 8, "reg_alpha": 0.30, "reg_lambda": 1.5,
        "subsample": 0.65, "colsample_bytree": 0.65,
        "n_estimators": 800
    }
}

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("training_audusd.log", encoding='utf-8'),
        logging.StreamHandler()
    ]
)


# ══════════════════════════════════════════════════════════════
# DATA FETCHER
# ══════════════════════════════════════════════════════════════
class TwelveDataFetcher:
    BASE_URL = "https://api.twelvedata.com/time_series"

    def __init__(self, api_key: str):
        self.api_key = api_key

    def fetch_range(self, symbol=SYMBOL, interval="5min",
                    start_date=None, end_date=None, outputsize=5000):
        params = {
            'symbol': symbol, 'interval': interval,
            'apikey': self.api_key, 'format': 'JSON',
            'order': 'ASC', 'outputsize': outputsize
        }
        if start_date: params['start_date'] = start_date
        if end_date:   params['end_date']   = end_date

        for attempt in range(5):
            try:
                r = requests.get(self.BASE_URL, params=params, timeout=30)
                r.raise_for_status()
                data = r.json()

                if 'code' in data and data['code'] == 429:
                    time.sleep(60); continue

                if 'values' not in data:
                    msg = data.get('message', '')
                    if "No data" in msg or "not available" in msg.lower():
                        logging.info(f"نهاية التاريخ للفريم {interval}")
                    else:
                        logging.error(f"خطأ API: {msg}")
                    return pd.DataFrame()

                df = pd.DataFrame(data['values'])
                if df.empty: return df
                if 'volume' not in df.columns: df['volume'] = 0
                for col in ['open','high','low','close','volume']:
                    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
                df['datetime'] = pd.to_datetime(df['datetime'])
                df.sort_values('datetime', inplace=True)
                df.reset_index(drop=True, inplace=True)
                return df

            except Exception as e:
                logging.warning(f"محاولة {attempt+1} فشلت: {e}")
                if attempt < 4:
                    time.sleep((2 ** attempt) + random.random())
        return pd.DataFrame()


def fetch_full_history(target_candles=100000, interval="5min", max_requests=25):
    fetcher = TwelveDataFetcher(TWELVE_DATA_API_KEY)
    all_data, end_date, batch_size = [], datetime.utcnow(), 5000

    mult = 60
    if "min" in interval:   mult = int(interval.replace("min","")) * 60
    elif interval.endswith("h"): mult = int(interval.replace("h","")) * 3600
    elif "d" in interval:   mult = 86400
    elif "week" in interval: mult = 604800

    for req in range(max_requests):
        start_date = end_date - timedelta(seconds=batch_size * mult)
        df_b = fetcher.fetch_range(
            symbol=SYMBOL, interval=interval,
            start_date=start_date.strftime("%Y-%m-%d %H:%M:%S"),
            end_date=end_date.strftime("%Y-%m-%d %H:%M:%S"),
            outputsize=batch_size
        )
        if df_b.empty: break
        all_data.append(df_b)
        logging.info(f"~{len(all_data)*batch_size} شمعة ({interval})")
        end_date = df_b['datetime'].min() - timedelta(seconds=1)
        if len(all_data) * batch_size >= target_candles: break
        time.sleep(0.6)

    if not all_data: return pd.DataFrame()
    df = pd.concat(all_data, ignore_index=True)
    df.drop_duplicates(subset='datetime', keep='first', inplace=True)
    df.sort_values('datetime', inplace=True)
    df.reset_index(drop=True, inplace=True)
    if len(df) > target_candles: df = df.iloc[-target_candles:]
    df.to_csv(f"{SYMBOL_SAFE}_{interval}_raw.csv", index=False)
    logging.info(f"✅ {len(df)} شمعة → {SYMBOL_SAFE}_{interval}_raw.csv")
    return df


def get_or_update_data(symbol_safe, interval="5min", target_candles=100000, force_refresh=False):
    csv_path = f"{symbol_safe}_{interval}_raw.csv"
    if os.path.exists(csv_path) and not force_refresh:
        df_old = pd.read_csv(csv_path)
        df_old['datetime'] = pd.to_datetime(df_old['datetime'])
        last_date = df_old['datetime'].max()
        fetcher = TwelveDataFetcher(TWELVE_DATA_API_KEY)
        df_new = fetcher.fetch_range(
            symbol=SYMBOL, interval=interval,
            start_date=(last_date + timedelta(seconds=1)).strftime("%Y-%m-%d %H:%M:%S"),
            end_date=datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),
            outputsize=1000
        )
        if df_new.empty: return df_old
        df = pd.concat([df_old, df_new], ignore_index=True)
        df.drop_duplicates(subset='datetime', keep='last', inplace=True)
        df.sort_values('datetime', inplace=True)
        df.to_csv(csv_path, index=False)
        return df
    max_req = 60 if interval in ["1h","4h","1d"] else 25
    return fetch_full_history(target_candles=target_candles, interval=interval, max_requests=max_req)


# ══════════════════════════════════════════════════════════════
# FEATURE ENGINEER
# ══════════════════════════════════════════════════════════════
class FeatureEngineer:

    def __init__(self):
        self.scaler          = RobustScaler()
        self.feature_columns = None
        self.scaler_fitted   = False

    def add_time_features(self, df):
        df = df.copy()
        df['hour']        = df['datetime'].dt.hour
        df['day_of_week'] = df['datetime'].dt.dayofweek
        df['month']       = df['datetime'].dt.month
        df['hour_sin']    = np.sin(2 * np.pi * df['hour'] / 24)
        df['hour_cos']    = np.cos(2 * np.pi * df['hour'] / 24)
        df['dow_sin']     = np.sin(2 * np.pi * df['day_of_week'] / 5)
        df['dow_cos']     = np.cos(2 * np.pi * df['day_of_week'] / 5)
        h = df['hour']
        # ★ AUD/USD sessions — Sydney أهم session لـ AUD
        # Sydney: 22:00-07:00 UTC (أهم لـ AUD)
        # Tokyo:  00:00-09:00 UTC (overlap مع Sydney)
        # London: 08:00-16:00 UTC
        # NY:     13:00-22:00 UTC
        df['is_sydney']   = ((h >= 22) | (h < 7)).astype(int)   # ★ AUD home session
        df['is_tokyo']    = ((h >= 0)  & (h < 9)).astype(int)   # ★ AUD + JPY session
        df['is_asia']     = ((h >= 0)  & (h < 8)).astype(int)   # general asia
        df['is_london']   = ((h >= 8)  & (h < 16)).astype(int)
        df['is_ny']       = ((h >= 13) & (h < 22)).astype(int)
        df['is_overlap']  = ((h >= 13) & (h < 17)).astype(int)
        df['is_friday']   = (df['day_of_week'] == 4).astype(int)
        df['is_monday']   = (df['day_of_week'] == 0).astype(int)
        df['is_tuesday']  = (df['day_of_week'] == 1).astype(int)  # ★ RBA meets Tuesdays
        # ★ AUD session weights: Sydney أعلى وزن
        df['session_momentum'] = (
            df['is_sydney']    * 3.0   # ★ AUD home session — الأهم
            + df['is_tokyo']   * 2.0   # ★ Asia overlap
            + df['is_london']  * 2.0
            + df['is_ny']      * 1.5
            + df['is_overlap'] * 3.0
            + df['is_asia']    * 1.0
        )
        # ★ RBA pre-decision risk (first Tuesday of month)
        df['is_rba_risk']  = (df['is_tuesday'] & (df['month'].isin([1,2,3,4,5,6,7,8,9,10,11,12]))).astype(int)
        # ★ China data hours (AUD commodity currency)
        df['is_china_data'] = ((h >= 1) & (h < 3)).astype(int)
        df['is_pre_news']   = ((h == 13) | (h == 14) | (h == 8) | (h == 1)).astype(int)
        return df

    def compute_indicators(self, df):
        df = df.copy()
        c, h, l, o, v = df['close'], df['high'], df['low'], df['open'], df['volume']
        n = len(df)

        p14  = min(14,  max(3, n // 6))
        p20  = min(20,  max(4, n // 5))
        p50  = min(50,  max(5, n // 4))
        p200 = min(200, max(10, n // 2))

        for span in [9, 21, p50, min(100, max(8, n // 3)), p200]:
            df[f'ema_{span}'] = c.ewm(span=span, adjust=False).mean()
        df['ema_9']   = c.ewm(span=9,   adjust=False).mean()
        df['ema_21']  = c.ewm(span=21,  adjust=False).mean()
        df['ema_50']  = c.ewm(span=p50, adjust=False).mean()
        df['ema_100'] = c.ewm(span=min(100, max(8, n // 3)), adjust=False).mean()
        df['ema_200'] = c.ewm(span=p200, adjust=False).mean()
        df['sma_20']  = c.rolling(p20).mean()
        df['sma_50']  = c.rolling(p50).mean()

        delta = c.diff()
        gain  = delta.clip(lower=0).rolling(p14).mean()
        loss  = (-delta.clip(upper=0)).rolling(p14).mean()
        df['rsi'] = 100 - (100 / (1 + gain / (loss + 1e-9)))

        rsi_min = df['rsi'].rolling(p14).min()
        rsi_max = df['rsi'].rolling(p14).max()
        df['stoch_rsi'] = (df['rsi'] - rsi_min) / (rsi_max - rsi_min + 1e-9)

        ema12 = c.ewm(span=min(12, max(3, n // 8)), adjust=False).mean()
        ema26 = c.ewm(span=min(26, max(5, n // 5)), adjust=False).mean()
        macd  = ema12 - ema26
        sig   = macd.ewm(span=min(9, max(3, n // 10)), adjust=False).mean()
        df['macd']        = macd
        df['macd_signal'] = sig
        df['macd_hist']   = macd - sig

        tr = pd.concat([h - l,
                        (h - c.shift(1)).abs(),
                        (l - c.shift(1)).abs()], axis=1).max(axis=1)
        df['atr']      = tr.rolling(p14).mean()
        df['atr_pct']  = df['atr'] / (c + 1e-9)
        df['atr_pips'] = df['atr'] / PIP_SIZE

        bb_mid = c.rolling(p20).mean()
        bb_std = c.rolling(p20).std()
        df['bb_upper'] = bb_mid + 2 * bb_std
        df['bb_lower'] = bb_mid - 2 * bb_std
        df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / (bb_mid + 1e-9)
        df['bb_pct']   = (c - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'] + 1e-9)

        tp = (h + l + c) / 3
        df['cci'] = (tp - tp.rolling(p20).mean()) / (0.015 * tp.rolling(p20).std() + 1e-9)

        hh14 = h.rolling(p14).max()
        ll14 = l.rolling(p14).min()
        df['williams_r'] = -100 * (hh14 - c) / (hh14 - ll14 + 1e-9)

        pos_mf = (tp * v).where(tp > tp.shift(1), 0).rolling(p14).sum()
        neg_mf = (tp * v).where(tp < tp.shift(1), 0).rolling(p14).sum()
        df['mfi'] = 100 - (100 / (1 + pos_mf / (neg_mf + 1e-9)))

        plus_dm  = (h - h.shift(1)).clip(lower=0)
        minus_dm = (l.shift(1) - l).clip(lower=0)
        plus_dm  = plus_dm.where(plus_dm  > minus_dm, 0)
        minus_dm = minus_dm.where(minus_dm > plus_dm,  0)
        atr14    = tr.rolling(p14).mean()
        pdi      = 100 * plus_dm.rolling(p14).mean()  / (atr14 + 1e-9)
        mdi      = 100 * minus_dm.rolling(p14).mean() / (atr14 + 1e-9)
        dx       = (pdi - mdi).abs() / (pdi + mdi + 1e-9) * 100
        df['adx']       = dx.rolling(p14).mean()
        df['adx_plus']  = pdi
        df['adx_minus'] = mdi

        s5  = min(5,  max(1, n // 15))
        s10 = min(10, max(2, n // 10))
        s20 = min(20, max(3, n // 7))
        df['momentum_5']  = c - c.shift(s5)
        df['momentum_10'] = c - c.shift(s10)
        df['roc_5']       = c.pct_change(s5)
        df['roc_10']      = c.pct_change(s10)
        df['roc_20']      = c.pct_change(s20)

        body   = (c - o).abs()
        range_ = (h - l).clip(lower=1e-9)
        df['body_ratio'] = body / range_
        df['upper_wick'] = (h - c.clip(lower=o))  / range_
        df['lower_wick'] = (c.clip(upper=o) - l)  / range_
        df['candle_dir'] = np.sign(c - o)

        df['vol_regime']       = c.pct_change().rolling(p20).std()
        df['vol_ratio']        = df['atr'] / (df['atr'].rolling(p50).mean() + 1e-9)
        df['daily_range_pips'] = (h - l) / PIP_SIZE
        df['pip_volatility']   = df['vol_regime'] / PIP_SIZE

        # ★ FIX-5: إضافة ميزات momentum بالـ pips (أوضح للموديل)
        df['momentum_5_pips']  = df['momentum_5']  / PIP_SIZE
        df['momentum_10_pips'] = df['momentum_10'] / PIP_SIZE

        return df

    def add_advanced_volume(self, df):
        df = df.copy()
        c, v, h, l = df['close'], df['volume'], df['high'], df['low']

        df['volume_delta']   = v * np.sign(c.diff())
        df['cvd']            = df['volume_delta'].cumsum()
        df['cvd_slope']      = df['cvd'].diff(5) / (df['cvd'].abs() + 1e-9)
        df['cvd_divergence'] = np.where(df['cvd_slope'] * c.pct_change() < 0, 1, 0)

        df['vpt']       = (v * c.pct_change()).cumsum()
        df['vpt_slope'] = df['vpt'].diff(5) / (df['vpt'].abs() + 1e-9)

        vol_ma20 = v.rolling(20).mean()
        vol_ma50 = v.rolling(50).mean()
        df['vol_ratio_20'] = v / (vol_ma20 + 1e-9)
        df['vol_ratio_50'] = v / (vol_ma50 + 1e-9)
        df['volume_surge'] = (v > vol_ma20 * 2).astype(int)

        bull_vol = v.where(c > c.shift(1), 0)
        bear_vol = v.where(c < c.shift(1), 0)
        df['vol_zone_bull'] = bull_vol.rolling(20).mean() / (vol_ma20 + 1e-9)
        df['vol_zone_bear'] = bear_vol.rolling(20).mean() / (vol_ma20 + 1e-9)
        df['vol_imbalance'] = (
            (bull_vol.rolling(10).sum() - bear_vol.rolling(10).sum())
            / (v.rolling(10).sum() + 1e-9)
        )

        price_up = (c.diff() > 0).astype(int)
        vol_up   = (v.diff() > 0).astype(int)
        df['volume_divergence'] = np.where(price_up != vol_up, 1, 0)

        df['eom']     = (
            ((h + l) / 2 - (h.shift(1) + l.shift(1)) / 2)
            / (v / (h - l + 1e-9) + 1e-9)
        )
        df['eom_sma'] = df['eom'].rolling(14).mean()
        return df

    def add_price_action(self, df):
        df = df.copy()
        o, h, l, c = df['open'], df['high'], df['low'], df['close']
        body       = (c - o).abs()
        range_     = (h - l + 1e-9)
        upper_wick = h - c.clip(lower=o)
        lower_wick = c.clip(upper=o) - l

        df['pinbar_bull'] = ((lower_wick / range_ > 0.6) & (body / range_ < 0.3)).astype(int)
        df['pinbar_bear'] = ((upper_wick / range_ > 0.6) & (body / range_ < 0.3)).astype(int)
        df['pinbar']      = (df['pinbar_bull'] | df['pinbar_bear']).astype(int)

        df['engulfing_bull'] = (
            (c.shift(1) < o.shift(1)) & (c > o)
            & (o <= c.shift(1)) & (c >= o.shift(1))
        ).astype(int)
        df['engulfing_bear'] = (
            (c.shift(1) > o.shift(1)) & (c < o)
            & (o >= c.shift(1)) & (c <= o.shift(1))
        ).astype(int)
        df['engulfing'] = (df['engulfing_bull'] | df['engulfing_bear']).astype(int)

        df['inside_bar']  = ((h < h.shift(1)) & (l > l.shift(1))).astype(int)
        df['outside_bar'] = ((h > h.shift(1)) & (l < l.shift(1))).astype(int)
        df['doji']        = (body / range_ < 0.1).astype(int)

        df['three_white_soldiers'] = (
            (c > o) & (c.shift(1) > o.shift(1)) & (c.shift(2) > o.shift(2))
            & (c > c.shift(1)) & (c.shift(1) > c.shift(2))
        ).astype(int)
        df['three_black_crows'] = (
            (c < o) & (c.shift(1) < o.shift(1)) & (c.shift(2) < o.shift(2))
            & (c < c.shift(1)) & (c.shift(1) < c.shift(2))
        ).astype(int)

        df['gap_up']      = (o > h.shift(1)).astype(int)
        df['gap_down']    = (o < l.shift(1)).astype(int)
        df['price_accel'] = c.diff().diff()
        return df

    def add_trend_filter(self, df):
        df = df.copy()
        c = df['close']
        df['trend_filter']     = (df['ema_50']  > df['ema_200']).astype(int)
        df['trend_filter_9']   = (df['ema_9']   > df['ema_21']).astype(int)
        df['ema_slope_50']     = df['ema_50'].diff(3)  / (c + 1e-9)
        df['ema_slope_200']    = df['ema_200'].diff(3) / (c + 1e-9)
        df['price_vs_ema50']   = (c > df['ema_50']).astype(int)
        df['price_vs_ema200']  = (c > df['ema_200']).astype(int)
        df['price_vs_ema9']    = (c > df['ema_9']).astype(int)
        df['trend_strength']   = (df['ema_50']  - df['ema_200']) / (c + 1e-9)
        df['trend_strength_9'] = (df['ema_9']   - df['ema_21'])  / (c + 1e-9)
        df['ema_alignment']    = (
            (df['ema_9']   > df['ema_21']).astype(int)
            + (df['ema_21']  > df['ema_50']).astype(int)
            + (df['ema_50']  > df['ema_100']).astype(int)
            + (df['ema_100'] > df['ema_200']).astype(int)
        ) / 4.0
        return df

    def add_market_structure(self, df, window=10):
        df = df.copy()
        high, low, c = df['high'], df['low'], df['close']
        rh = high.rolling(window).max()
        rl = low.rolling(window).min()
        prh = rh.shift(window)
        prl = rl.shift(window)

        df['hh'] = (rh > prh).astype(int)
        df['hl'] = (rl > prl).astype(int)
        df['lh'] = (rh < prh).astype(int)
        df['ll'] = (rl < prl).astype(int)
        df['market_structure'] = np.where(
            (df['hh'] == 1) & (df['hl'] == 1), 1,
            np.where((df['lh'] == 1) & (df['ll'] == 1), -1, 0)
        )
        df['bos_up']    = (c > rh.shift(1)).astype(int)
        df['bos_down']  = (c < rl.shift(1)).astype(int)
        df['swing_high'] = (
            (high > high.shift(1)) & (high > high.shift(-1))
            & (high > high.shift(2)) & (high > high.shift(-2))
        ).astype(int)
        df['swing_low'] = (
            (low < low.shift(1)) & (low < low.shift(-1))
            & (low < low.shift(2)) & (low < low.shift(-2))
        ).astype(int)
        return df

    def add_support_resistance(self, df, window=20):
        df = df.copy()
        c  = df['close']
        w   = min(window, max(5, len(df) // 6))
        w50 = min(50, max(5, len(df) // 4))

        df['resistance']         = df['high'].rolling(w).max()
        df['support']            = df['low'].rolling(w).min()
        df['dist_to_resistance'] = (df['resistance'] - c) / (c + 1e-9)
        df['dist_to_support']    = (c - df['support'])    / (c + 1e-9)
        # ★ AUD: threshold نفس EUR/USD (pip size نفسه، 0.0001)
        df['near_resistance']    = (df['dist_to_resistance'] < 0.0005).astype(int)
        df['near_support']       = (df['dist_to_support']    < 0.0005).astype(int)

        sr_range      = df['resistance'] - df['support']
        df['sr_dist'] = (c - df['support']) / (sr_range + 1e-9)
        df['resistance_50'] = df['high'].rolling(w50).max()
        df['support_50']    = df['low'].rolling(w50).min()
        df['sr_dist_50']    = (c - df['support_50']) / (
            df['resistance_50'] - df['support_50'] + 1e-9
        )
        df['dist_to_resistance_pips'] = df['dist_to_resistance'] * c / PIP_SIZE
        df['dist_to_support_pips']    = df['dist_to_support']    * c / PIP_SIZE
        return df

    def add_volume_profile(self, df):
        df = df.copy()
        c, h, l, v = df['close'], df['high'], df['low'], df['volume']
        tp = (h + l + c) / 3

        vwap_window = min(144, max(5, len(df) // 4))
        obv_window  = min(20,  max(3, len(df) // 10))

        vol_sum    = v.rolling(vwap_window).sum() + 1e-9
        df['vwap'] = (tp * v).rolling(vwap_window).sum() / vol_sum

        df['dist_to_vwap']  = (c - df['vwap']) / (df['vwap'] + 1e-9)
        df['price_vs_vwap'] = (c > df['vwap']).astype(int)

        vwap_std          = (tp - df['vwap']).rolling(vwap_window).std()
        df['vwap_upper1'] = df['vwap'] + 1.0 * vwap_std
        df['vwap_lower1'] = df['vwap'] - 1.0 * vwap_std
        df['vwap_upper2'] = df['vwap'] + 2.0 * vwap_std
        df['vwap_lower2'] = df['vwap'] - 2.0 * vwap_std
        df['vwap_band_pct'] = (c - df['vwap_lower2']) / (
            df['vwap_upper2'] - df['vwap_lower2'] + 1e-9
        )

        obv_dir         = np.sign(c.diff())
        df['obv']       = (v * obv_dir).cumsum()
        df['obv_sma']   = df['obv'].rolling(obv_window).mean()
        df['obv_slope'] = df['obv'].diff(min(5, len(df) // 8)) / (df['obv'].abs() + 1e-9)
        df['obv_vs_sma'] = (df['obv'] > df['obv_sma']).astype(int)
        return df

    def add_digital_indicators(self, df):
        df = df.copy()
        c, h, l = df['close'], df['high'], df['low']
        n = len(df)

        t9  = min(9,  max(3, n // 8))
        t26 = min(26, max(5, n // 5))
        t52 = min(52, max(8, n // 3))
        tenkan   = (h.rolling(t9).max()  + l.rolling(t9).min())  / 2
        kijun    = (h.rolling(t26).max() + l.rolling(t26).min()) / 2
        shift26  = min(26, max(3, n // 6))
        senkou_a = ((tenkan + kijun) / 2).shift(shift26)
        senkou_b = ((h.rolling(t52).max() + l.rolling(t52).min()) / 2).shift(shift26)

        df['ichi_tenkan']         = tenkan
        df['ichi_kijun']          = kijun
        df['ichi_senkou_a']       = senkou_a
        df['ichi_senkou_b']       = senkou_b
        df['ichi_cloud_bull']     = (senkou_a > senkou_b).astype(int)
        df['ichi_above_cloud']    = ((c > senkou_a) & (c > senkou_b)).astype(int)
        df['ichi_tk_cross']       = (
            (tenkan > kijun) & (tenkan.shift(1) <= kijun.shift(1))
        ).astype(int)
        df['ichi_price_vs_kijun'] = (c > kijun).astype(int)

        cmo_p = min(14, max(3, n // 6))
        d_ = c.diff()
        up_ = d_.clip(lower=0).rolling(cmo_p).sum()
        dn_ = (-d_.clip(upper=0)).rolling(cmo_p).sum()
        df['cmo'] = 100 * (up_ - dn_) / (up_ + dn_ + 1e-9)

        er_period = min(10, max(3, n // 8))
        fast_sc, slow_sc = 2/(2+1), 2/(30+1)
        direction  = c.diff(er_period).abs()
        volatility = c.diff().abs().rolling(er_period).sum()
        er = direction / (volatility + 1e-9)
        sc = (er * (fast_sc - slow_sc) + slow_sc) ** 2
        kama_arr = c.to_numpy(dtype=float).copy()
        for i in range(er_period, len(kama_arr)):
            kama_arr[i] = kama_arr[i-1] + sc.iloc[i] * (kama_arr[i] - kama_arr[i-1])
        df['kama']          = kama_arr
        df['price_vs_kama'] = (c > df['kama']).astype(int)
        df['kama_slope']    = pd.Series(kama_arr, index=c.index).diff(min(3, max(1, n//12))) / (c + 1e-9)

        zl_p     = min(21, max(5, n // 5))
        lag_     = (zl_p - 1) // 2
        zl_input = c + (c - c.shift(lag_))
        df['zlema']          = zl_input.ewm(span=zl_p, adjust=False).mean()
        df['price_vs_zlema'] = (c > df['zlema']).astype(int)
        df['zlema_slope']    = df['zlema'].diff(min(3, max(1, n//12))) / (c + 1e-9)

        fd_n = min(20, max(4, n // 5))
        df['fractal_dim'] = (h.rolling(fd_n).max() - l.rolling(fd_n).min()) / (
            df['atr'] * np.sqrt(fd_n) + 1e-9
        )
        return df

    def add_htf_alignment(self, df):
        df = df.copy()
        idx = df.index

        def _g(col):
            return df[col] if col in df.columns else pd.Series(0.0, index=idx)

        trend     = _g('trend_filter')
        ms        = _g('market_structure').clip(-1, 1) * 0.5
        cvd       = _g('cvd_slope').clip(-1, 1)
        vol_div   = _g('volume_divergence')
        adx_v     = (_g('adx') / 100).clip(0, 1)
        ema_align = _g('ema_alignment')
        session   = _g('session_momentum') / 4.0

        df['confirmation_score'] = (
            0.25 * trend
            + 0.20 * ms
            + 0.12 * cvd
            + 0.08 * (1 - vol_div)
            + 0.15 * adx_v
            + 0.10 * ema_align
            + 0.10 * session
        )
        df['signal_strength'] = (
            df['confirmation_score']          * 0.4
            + _g('trend_strength').clip(-1, 1) * 0.3
            + _g('vol_imbalance').clip(-1, 1)  * 0.3
        )
        return df

    # ★ FIX-6: إضافة ميزات regime detection
    def add_regime_features(self, df):
        df = df.copy()
        c = df['close']
        n = len(df)

        # Volatility regime
        p20 = min(20, max(4, n // 5))
        p50 = min(50, max(5, n // 4))
        current_vol = c.pct_change().rolling(p20).std()
        hist_vol    = c.pct_change().rolling(p50).std()
        df['vol_regime_ratio'] = current_vol / (hist_vol + 1e-9)
        # ★ AUD: volatility thresholds مختلفة
        # AUD/USD أهدأ من EUR/USD → thresholds أضيق
        df['is_high_vol'] = (df['vol_regime_ratio'] > 1.3).astype(int)
        df['is_low_vol']  = (df['vol_regime_ratio'] < 0.75).astype(int)

        # Trend strength via ADX buckets
        if 'adx' in df.columns:
            df['adx_strong']  = (df['adx'] > 25).astype(int)
            df['adx_ranging'] = (df['adx'] < 20).astype(int)

        # Distance from recent high/low as percent (mean reversion signal)
        w = min(20, max(5, n // 6))
        df['pct_from_high'] = (df['high'].rolling(w).max() - c) / (c + 1e-9)
        df['pct_from_low']  = (c - df['low'].rolling(w).min())  / (c + 1e-9)

        # RSI divergence
        if 'rsi' in df.columns:
            df['rsi_overbought']  = (df['rsi'] > 70).astype(int)
            df['rsi_oversold']    = (df['rsi'] < 30).astype(int)
            df['rsi_mid_bullish'] = ((df['rsi'] > 50) & (df['rsi'] <= 70)).astype(int)
            df['rsi_mid_bearish'] = ((df['rsi'] < 50) & (df['rsi'] >= 30)).astype(int)

        return df

    def create_features(self, df, shift=True):
        df = self.add_time_features(df)
        df = self.compute_indicators(df)
        df = self.add_advanced_volume(df)
        df = self.add_price_action(df)
        df = self.add_trend_filter(df)
        df = self.add_market_structure(df)
        df = self.add_support_resistance(df)
        df = self.add_volume_profile(df)
        df = self.add_digital_indicators(df)
        df = self.add_htf_alignment(df)
        df = self.add_regime_features(df)   # ★ FIX-6

        if shift:
            exclude_shift = {'datetime','open','high','low','close','volume'}
            shift_cols = [col for col in df.columns if col not in exclude_shift]
            df[shift_cols] = df[shift_cols].shift(1)

        df = df.ffill().fillna(0)
        df.replace([np.inf, -np.inf], 0, inplace=True)

        warmup = max(10, min(60, len(df) // 10))
        df = df.iloc[warmup:].copy()

        thresh = int(len(df) * 0.3)
        df = df.dropna(axis=1, thresh=thresh)
        df = df.ffill().fillna(0)
        df.replace([np.inf, -np.inf], 0, inplace=True)

        exclude_final = {
            'datetime','open','high','low','close','volume',
            'future_close','pct_change','target'
        }
        self.feature_columns = [col for col in df.columns if col not in exclude_final]
        return df

    # ★ FIX-1: create_target مع pip threshold
    def create_target(self, df, horizon=1, pip_threshold=0):
        df = df.copy()

        # أقصى high في الـ horizon شموع القادمة
        df['future_high']  = df['high'].rolling(horizon).max().shift(-horizon)
        df['future_low']   = df['low'].rolling(horizon).min().shift(-horizon)
        df['future_close'] = df['close'].shift(-horizon)

        move_up   = (df['future_high']  - df['close']) / PIP_SIZE
        move_down = (df['close'] - df['future_low'])   / PIP_SIZE

        if pip_threshold > 0:
            # 1 = buy (السعر ارتفع كفاية), 0 = sell/flat
            df['target'] = np.where(
                (move_up > pip_threshold) & (move_up > move_down), 1,
                np.where((move_down > pip_threshold) & (move_down > move_up), 0, -1)
            )
            # حذف الـ neutral (-1) — مش عارفين الاتجاه
            df = df[df['target'] != -1].copy()
        else:
            df['target'] = (df['future_close'] > df['close']).astype(int)

        df['pct_change'] = (df['future_close'] - df['close']) / (df['close'] + 1e-9)
        df.dropna(subset=['future_close'], inplace=True)
        df['target'] = df['target'].astype(int)
        return df

    def fit_scaler(self, X):
        self.scaler.fit(X)
        self.scaler_fitted = True
        if isinstance(X, pd.DataFrame):
            self.feature_columns = list(X.columns)

    def transform(self, X):
        if not self.scaler_fitted:
            raise RuntimeError("يجب تدريب الـ scaler أولاً")
        if isinstance(X, pd.DataFrame):
            for col in self.feature_columns:
                if col not in X.columns: X[col] = 0.0
            X = X[self.feature_columns]
        return self.scaler.transform(X)

    def fit_transform(self, X):
        self.fit_scaler(X)
        return self.transform(X)


# ══════════════════════════════════════════════════════════════
# MULTI-TIMEFRAME BUILDER
# ══════════════════════════════════════════════════════════════
class MultiTimeframeBuilder:
    def __init__(self):
        self._htf_cache = {}

    def _build_htf_df(self, df_raw, resample_rule):
        if resample_rule in self._htf_cache:
            return self._htf_cache[resample_rule]
        df_htf = (
            df_raw.set_index('datetime')
            .resample(resample_rule)
            .agg({'open':'first','high':'max','low':'min','close':'last','volume':'sum'})
            .dropna().reset_index()
        )
        fe_htf   = FeatureEngineer()
        df_htf_f = fe_htf.create_features(df_htf, shift=True)
        df_htf_f = fe_htf.create_target(df_htf_f, horizon=1)
        self._htf_cache[resample_rule] = df_htf_f
        return df_htf_f

    def merge_htf_features(self, df_ltf, df_raw, htf_resample_rule,
                           htf_feature_names, htf_label):
        if not htf_feature_names or htf_resample_rule is None:
            return df_ltf
        df_htf    = self._build_htf_df(df_raw, htf_resample_rule)
        available = [c for c in htf_feature_names if c in df_htf.columns]
        if not available: return df_ltf
        df_sel = df_htf[['datetime'] + available].copy()
        df_sel.columns = ['datetime'] + [f'htf_{htf_label}_{c}' for c in available]
        df_merged = pd.merge_asof(
            df_ltf.sort_values('datetime'),
            df_sel.sort_values('datetime'),
            on='datetime', direction='backward'
        )
        return df_merged.ffill().fillna(0)


# ══════════════════════════════════════════════════════════════
# BACKTESTER — محسّن لـ EUR/USD
# ★ FIX-7: إضافة trailing stop + minimum pip move للدخول
# ══════════════════════════════════════════════════════════════
class Backtester:
    def __init__(self, initial_capital=10000, spread_bps=SPREAD_BPS, risk_pct=0.02):
        # ★ AUD: نفس EUR/USD — spread أعلى قليلاً لكن risk_pct نفسه
        self.initial_capital = initial_capital
        self.spread_bps      = spread_bps / 10000
        self.slippage        = 0.00005
        self.risk_pct        = risk_pct

    def _exec_price(self, price, is_buy):
        if is_buy:
            return price * (1 + self.spread_bps) * (1 + self.slippage)
        return price * (1 - self.spread_bps) * (1 - self.slippage)

    def run(self, df, model, fe, feature_columns, timeframe_config,
            atr_mult_stop=ATR_MULT_STOP, atr_mult_take=ATR_MULT_TAKE,
            use_fold_thresholds=False):

        capital   = self.initial_capital
        positions = []
        trades    = []
        equity    = [capital]

        a_stop = timeframe_config.get('atr_mult_stop', atr_mult_stop)
        a_take = timeframe_config.get('atr_mult_take', atr_mult_take)

        missing = [c for c in feature_columns + ['close','atr']
                   if c not in df.columns]
        if missing:
            logging.error(f"أعمدة مفقودة: {missing}")
            return {}

        key_buy  = 'fold_buy_threshold'  if use_fold_thresholds else 'buy_threshold'
        key_sell = 'fold_sell_threshold' if use_fold_thresholds else 'sell_threshold'
        buy_th   = timeframe_config.get(key_buy,  0.57)
        sell_th  = timeframe_config.get(key_sell, 0.43)

        max_daily    = timeframe_config.get('max_trades_per_day')
        daily_counts = {}

        X_scaled  = fe.transform(df[feature_columns].copy())
        all_proba = model.predict_proba(X_scaled)[:, 1]
        closes    = df['close'].values
        atrs      = df['atr'].values
        datetimes = df['datetime'].values if 'datetime' in df.columns else [None] * len(df)

        for i in range(len(df)):
            p, atr, prob = closes[i], max(float(atrs[i]), 1e-6), all_proba[i]

            signal = "HOLD"
            if prob > buy_th:  signal = "BUY"
            elif prob < sell_th: signal = "SELL"

            if signal == "BUY" and max_daily and datetimes[i] is not None:
                dk = str(datetimes[i])[:10]
                if daily_counts.get(dk, 0) >= max_daily: signal = "HOLD"

            if signal == "BUY" and not positions:
                ep    = self._exec_price(p, True)
                units = capital * self.risk_pct / ep
                positions.append({
                    'entry_price': ep, 'units': units,
                    'stop_loss':   ep - a_stop * atr,
                    'take_profit': ep + a_take * atr,
                    'peak_price':  ep   # ★ للـ trailing stop
                })
                capital -= units * ep
                if max_daily and datetimes[i] is not None:
                    dk = str(datetimes[i])[:10]
                    daily_counts[dk] = daily_counts.get(dk, 0) + 1

            elif signal == "SELL" and positions:
                pos = positions.pop(0)
                ep  = self._exec_price(p, False)
                pnl = pos['units'] * (ep - pos['entry_price'])
                capital += pos['units'] * ep
                trades.append({
                    'profit': pnl,
                    'return_pct': pnl / (pos['units'] * pos['entry_price'] + 1e-9),
                    'exit_reason': 'signal'
                })

            if positions:
                pos = positions[0]
                # ★ FIX-7: trailing stop — بنتبع الـ peak
                if p > pos['peak_price']:
                    pos['peak_price'] = p
                    trail_stop = p - a_stop * atr
                    if trail_stop > pos['stop_loss']:
                        pos['stop_loss'] = trail_stop

                if p <= pos['stop_loss']:
                    ep  = pos['stop_loss']
                    pnl = pos['units'] * (ep - pos['entry_price'])
                    capital += pos['units'] * ep
                    trades.append({
                        'profit': pnl,
                        'return_pct': pnl / (pos['units'] * pos['entry_price'] + 1e-9),
                        'exit_reason': 'stop_loss'
                    })
                    positions.pop(0)
                elif p >= pos['take_profit']:
                    ep  = pos['take_profit']
                    pnl = pos['units'] * (ep - pos['entry_price'])
                    capital += pos['units'] * ep
                    trades.append({
                        'profit': pnl,
                        'return_pct': pnl / (pos['units'] * pos['entry_price'] + 1e-9),
                        'exit_reason': 'take_profit'
                    })
                    positions.pop(0)

            mtm = positions[0]['units'] * p if positions else 0
            equity.append(capital + mtm)

        if positions:
            pos = positions.pop(0)
            ep  = self._exec_price(closes[-1], False)
            pnl = pos['units'] * (ep - pos['entry_price'])
            capital += pos['units'] * ep
            trades.append({
                'profit': pnl,
                'return_pct': pnl / (pos['units'] * pos['entry_price'] + 1e-9),
                'exit_reason': 'end_of_data'
            })

        if not trades: return {}

        eq   = pd.Series(equity)
        wins = [t for t in trades if t['profit'] > 0]
        loss = [t for t in trades if t['profit'] <= 0]
        wr   = len(wins) / len(trades) * 100
        tp   = sum(t['profit'] for t in wins)
        tl   = abs(sum(t['profit'] for t in loss))
        dd   = ((eq - eq.cummax()) / (eq.cummax() + 1e-9) * 100).min()
        aw   = tp / len(wins) if wins else 0
        al   = tl / len(loss) if loss else 0

        return {
            'total_trades':     len(trades),
            'win_rate':         round(wr, 2),
            'profit_factor':    round(tp / tl, 2) if tl > 0 else float('inf'),
            'total_return_pct': round((capital - self.initial_capital) / self.initial_capital * 100, 2),
            'max_drawdown_pct': round(dd, 2),
            'avg_win':          round(aw, 2),
            'avg_loss':         round(al, 2),
            'expectancy':       round((wr / 100 * aw) - ((1 - wr / 100) * al), 2) if al else 0
        }


# ══════════════════════════════════════════════════════════════
# HELPER: Single Fold
# ══════════════════════════════════════════════════════════════
def _train_single_fold(X, y, xgb_param, timeframe):
    neg = (y == 0).sum()
    pos = (y == 1).sum()
    spw = neg / pos if pos > 0 else 1.0
    n_est = xgb_param.pop('n_estimators', 800)

    fe_fold = FeatureEngineer()
    fe_fold.fit_scaler(X)
    X_s = fe_fold.transform(X)

    model = xgb.XGBClassifier(
        objective='binary:logistic',
        n_estimators=n_est,
        scale_pos_weight=spw,
        random_state=42, tree_method='hist',
        eval_metric='logloss', verbosity=0,
        **xgb_param
    )
    model.fit(X_s, y, verbose=False)
    xgb_param['n_estimators'] = n_est
    logging.info(f"[{timeframe}] ✅ Single-fold — {len(X)} عينة")
    return model, fe_fold


# ══════════════════════════════════════════════════════════════
# WALK-FORWARD TRAINING
# ══════════════════════════════════════════════════════════════
def train_model_with_walk_forward(df_raw, timeframe, n_splits=5, mtf_builder=None):
    config        = TIMEFRAME_CONFIG[timeframe]
    resample_rule = config['resample_rule']
    years         = config['years']
    min_candles   = config.get('min_candles', 100)
    larger_tf     = config.get('larger_tf')
    htf_features  = config.get('htf_features', [])
    horizon       = HORIZON.get(timeframe, 1)
    pip_thr       = PIP_THRESHOLD.get(timeframe, 0)

    logging.info("=" * 60)
    logging.info(f"Walk-Forward [AUD/USD] — {timeframe} | horizon={horizon} | pip_thr={pip_thr}")
    logging.info("=" * 60)

    df_tf = (
        df_raw.set_index('datetime')
        .resample(resample_rule)
        .agg({'open':'first','high':'max','low':'min','close':'last','volume':'sum'})
        .dropna().reset_index()
    )
    if 'datetime' not in df_tf.columns:
        df_tf.rename(columns={'index':'datetime'}, inplace=True)

    if years:
        cutoff = df_tf['datetime'].max() - timedelta(days=365 * years)
        df_tf  = df_tf[df_tf['datetime'] >= cutoff].copy()

    logging.info(f"شموع بعد القطع: {len(df_tf)}")

    if len(df_tf) < min_candles:
        logging.error(f"❌ {timeframe}: {len(df_tf)} < {min_candles}")
        return None

    fe        = FeatureEngineer()
    df_feat   = fe.create_features(df_tf, shift=True)

    # ★ FIX-1: استخدام pip_threshold في create_target
    df_target = fe.create_target(df_feat, horizon=horizon, pip_threshold=pip_thr)

    if df_target.empty or len(df_target) < min_candles:
        logging.error(f"❌ {timeframe}: بيانات غير كافية بعد target ({len(df_target)})")
        return None

    # class balance check
    class_counts = df_target['target'].value_counts()
    logging.info(f"Class balance: {class_counts.to_dict()}")
    if len(class_counts) < 2:
        logging.error(f"❌ {timeframe}: class واحدة فقط بعد الـ pip threshold")
        return None

    if mtf_builder and larger_tf and htf_features:
        htf_cfg = TIMEFRAME_CONFIG.get(larger_tf)
        if htf_cfg:
            df_target = mtf_builder.merge_htf_features(
                df_ltf=df_target, df_raw=df_raw,
                htf_resample_rule=htf_cfg['resample_rule'],
                htf_feature_names=htf_features,
                htf_label=larger_tf
            )

    exclude_final = {
        'datetime','open','high','low','close','volume',
        'future_close','future_high','future_low','pct_change','target'
    }
    fe.feature_columns = [c for c in df_target.columns if c not in exclude_final]
    logging.info(f"Features: {len(fe.feature_columns)}")

    X = df_target[fe.feature_columns]
    y = df_target['target']

    xgb_param = dict(XGB_PARAMS.get(timeframe, XGB_PARAMS["default"]))
    n_est_cfg = xgb_param.pop('n_estimators', 800)

    if len(X) < 100:
        min_train = max(15, len(X) // 4)
        min_test  = max(5,  len(X) // 10)
        n_splits  = min(n_splits, 3)
    else:
        min_train = max(30, len(X) // (n_splits + 1))
        min_test  = max(8,  len(X) // (n_splits * 3))

    effective_splits = min(n_splits, max(2, len(X) // (min_train + min_test)))
    tscv = TimeSeriesSplit(n_splits=effective_splits)

    fold_metrics = []
    best_model   = None
    best_fe_fold = None
    best_score   = -np.inf
    last_model   = None
    last_fe_fold = None

    for fold, (tr_idx, te_idx) in enumerate(tscv.split(X)):
        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

        if len(X_tr) < min_train or len(X_te) < min_test:
            logging.warning(f"Fold {fold+1}: train={len(X_tr)} test={len(X_te)} — تخطي")
            continue

        neg = (y_tr == 0).sum()
        pos = (y_tr == 1).sum()
        if pos == 0 or neg == 0:
            logging.warning(f"Fold {fold+1}: class واحدة — تخطي")
            continue
        spw = neg / pos

        fe_fold = FeatureEngineer()
        fe_fold.fit_scaler(X_tr)
        X_tr_s = fe_fold.transform(X_tr)
        X_te_s = fe_fold.transform(X_te)

        # ★ FIX-4: n_estimators من الـ config + early stopping
        model = xgb.XGBClassifier(
            objective='binary:logistic',
            n_estimators=n_est_cfg,
            early_stopping_rounds=80,
            scale_pos_weight=spw,
            random_state=42, tree_method='hist',
            eval_metric='logloss', verbosity=0,
            **xgb_param
        )
        model.fit(X_tr_s, y_tr, eval_set=[(X_te_s, y_te)], verbose=False)

        last_model   = model
        last_fe_fold = fe_fold

        y_pred    = model.predict(X_te_s)
        acc       = accuracy_score(y_te, y_pred)
        precision = precision_score(y_te, y_pred, zero_division=0)
        recall    = recall_score(y_te, y_pred, zero_division=0)
        f1        = f1_score(y_te, y_pred, zero_division=0)

        df_test      = df_target.iloc[te_idx].copy()
        backtester   = Backtester()
        metrics_test = backtester.run(
            df_test, model, fe_fold,
            fe_fold.feature_columns, config,
            use_fold_thresholds=True
        )

        logging.info(
            f"  Fold {fold+1} — Acc:{acc:.2%}  F1:{f1:.2%}  "
            f"Prec:{precision:.2%}  Rec:{recall:.2%}"
        )
        if metrics_test:
            logging.info(
                f"  BT — WR:{metrics_test['win_rate']}%  "
                f"PF:{metrics_test['profit_factor']}  "
                f"Ret:{metrics_test['total_return_pct']}%  "
                f"Trades:{metrics_test['total_trades']}"
            )

        fold_metrics.append({
            'fold': fold+1, 'accuracy': acc, 'f1': f1,
            'precision': precision, 'recall': recall,
            'backtest': metrics_test
        })

        # Score مركب
        if metrics_test and metrics_test.get('total_trades', 0) >= 3:
            score = (
                metrics_test.get('profit_factor', 0) * 0.45
                + metrics_test.get('win_rate',    0) * 0.20
                + min(metrics_test.get('total_trades', 0), 50) * 0.25
                + f1 * 100                                      * 0.10
            )
        else:
            score = f1 * 100

        if score > best_score:
            best_score   = score
            best_model   = model
            best_fe_fold = fe_fold

    if best_model is None:
        logging.warning(f"⚠️  [{timeframe}] → Single Fold Training")
        xgb_param['n_estimators'] = n_est_cfg
        best_model, best_fe_fold = _train_single_fold(X, y, xgb_param, timeframe)
        xgb_param.pop('n_estimators', None)

    # Final model على كل البيانات
    neg_f = (y == 0).sum()
    pos_f = (y == 1).sum()
    n_est_final = n_est_cfg
    if hasattr(best_model, 'best_iteration') and best_model.best_iteration:
        n_est_final = max(100, best_model.best_iteration + 1)

    fe_full = FeatureEngineer()
    fe_full.fit_scaler(X)
    X_full_s = fe_full.transform(X)

    final_model = xgb.XGBClassifier(
        objective='binary:logistic',
        n_estimators=n_est_final,
        scale_pos_weight=neg_f / pos_f if pos_f else 1,
        random_state=42, tree_method='hist',
        eval_metric='logloss', verbosity=0,
        **xgb_param
    )
    final_model.fit(X_full_s, y)

    os.makedirs(MODELS_DIR, exist_ok=True)
    model_path = f"{MODELS_DIR}/{SYMBOL_SAFE}_{timeframe}.pkl"
    joblib.dump({
        'model':           final_model,
        'scaler':          fe_full.scaler,
        'feature_columns': fe_full.feature_columns,
        'config':          config,
        'horizon':         horizon,
        'pip_threshold':   pip_thr
    }, model_path)

    holdout_start = int(len(df_target) * 0.8)
    df_holdout    = df_target.iloc[holdout_start:].copy()
    full_metrics  = {}

    if len(df_holdout) >= 10:
        bt_holdout   = Backtester()
        full_metrics = bt_holdout.run(
            df_holdout, final_model, fe_full,
            fe_full.feature_columns, config
        )
        if full_metrics:
            logging.info(
                f"✅ Holdout — WR:{full_metrics['win_rate']}%  "
                f"PF:{full_metrics['profit_factor']}  "
                f"Ret:{full_metrics['total_return_pct']}%  "
                f"Trades:{full_metrics['total_trades']}"
            )

    metrics_file = f"{MODELS_DIR}/{SYMBOL_SAFE}_{timeframe}_metrics.json"
    with open(metrics_file, 'w', encoding='utf-8') as f:
        json.dump({
            'symbol': SYMBOL, 'timeframe': timeframe,
            'n_features': len(fe_full.feature_columns),
            'horizon': horizon, 'pip_threshold': pip_thr,
            'fold_results': fold_metrics,
            'holdout_backtest_last_20pct': full_metrics,
            'config': config
        }, f, indent=2, default=str, ensure_ascii=False)

    logging.info(f"💾 {model_path}")
    return final_model, fe_full


# ══════════════════════════════════════════════════════════════
# TRAIN ALL MODELS
# ══════════════════════════════════════════════════════════════
def train_all_models(force_refresh_data=False):
    logging.info("=" * 70)
    logging.info("★ AUD/USD Model Factory — Sydney Session + Commodity Currency ★")
    logging.info("=" * 70)

    df_5min = get_or_update_data(SYMBOL_SAFE, interval="5min", force_refresh=force_refresh_data)
    df_1h   = get_or_update_data(SYMBOL_SAFE, interval="1h",   force_refresh=force_refresh_data)

    mtf_builder = MultiTimeframeBuilder()
    results     = {}

    for tf in TIMEFRAME_CONFIG:
        logging.info("-" * 50)
        try:
            source_df = df_1h if tf in ["1h","2h","4h","1d","1W"] else df_5min
            if source_df is None or source_df.empty:
                results[tf] = "❌ فشل البيانات"
                continue
            res = train_model_with_walk_forward(source_df, tf, n_splits=5, mtf_builder=mtf_builder)
            results[tf] = "✅ نجح" if res else "❌ فشل"
        except Exception as e:
            logging.error(f"❌ {tf}: {e}", exc_info=True)
            results[tf] = f"❌ خطأ: {e}"

    logging.info("=" * 70)
    logging.info("ملخص:")
    for tf, st in results.items():
        logging.info(f"   {tf:6s}: {st}")
    logging.info(f"📁 {MODELS_DIR}/")


if __name__ == "__main__":
    train_all_models(force_refresh_data=True)

In [2]:
import os, shutil
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

ZIP_NAME = "audusd_models.zip"
if not os.path.exists(MODELS_DIR):
    print(f"❌ {MODELS_DIR} مش موجود — شغّل الـ training cell الأول")
else:
    fs = os.listdir(MODELS_DIR)
    print(f"📁 {MODELS_DIR}: {len([f for f in fs if f.endswith('.pkl')])} pkl | {len([f for f in fs if f.endswith('.json')])} json")
    for f in sorted(fs):
        size = os.path.getsize(os.path.join(MODELS_DIR, f))
        print(f"   {f:<45} {size/1024:.1f} KB")
    shutil.make_archive(ZIP_NAME.replace(".zip",""), "zip", ".", MODELS_DIR)
    mb = os.path.getsize(ZIP_NAME)/(1024*1024)
    print(f"\n✅ {ZIP_NAME} ({mb:.1f} MB)")
    if IN_COLAB: files.download(ZIP_NAME)
    else: print(f"📂 {os.path.abspath(ZIP_NAME)}")


📁 models/audusd: 8 pkl | 8 json
   AUDUSD_15min.pkl                              150.6 KB
   AUDUSD_15min_metrics.json                     2.5 KB
   AUDUSD_1W.pkl                                 975.7 KB
   AUDUSD_1W_metrics.json                        2.1 KB
   AUDUSD_1d.pkl                                 181.4 KB
   AUDUSD_1d_metrics.json                        2.0 KB
   AUDUSD_1h.pkl                                 205.7 KB
   AUDUSD_1h_metrics.json                        2.5 KB
   AUDUSD_2h.pkl                                 305.6 KB
   AUDUSD_2h_metrics.json                        2.5 KB
   AUDUSD_45min.pkl                              176.3 KB
   AUDUSD_45min_metrics.json                     2.2 KB
   AUDUSD_4h.pkl                                 375.0 KB
   AUDUSD_4h_metrics.json                        2.6 KB
   AUDUSD_5min.pkl                               168.0 KB
   AUDUSD_5min_metrics.json                      2.6 KB

✅ audusd_models.zip (0.7 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>